# Music Annotation Project V8.7
## Notebook 49R4: Connected-Subregion Coherence, Separability and M4 Semantics Audit

**Runtime:** CPU. This is a lightweight, read-only continuation of a completed Notebook 49 run.

### Purpose
1. Reuse the completed Notebook 49 pixel and region artefacts. Do not rebuild Notebook 45 inventories, masks, observable manifests or truth labels.
2. Split truth-side `REMOVE`, `PROTECT` and `UNCERTAIN` pixels into page-global 8-connected subregions within M3.
3. Quantify connected-subregion coherence, fragmentation, size, parent-component mixture and source-group coverage.
4. Quantify observable-only univariate separability for REMOVE and PROTECT subregions without fitting or selecting an M5 policy.
5. Diagnose M4 set semantics against M3, the Notebook 49 classes, clean printed ink, observed dark material, the authoritative mask and Notebook 45 arm identities.

### Interpretation boundary
- This notebook does not train, calibrate or select a classifier.
- Truth-side classes are outcomes, never inference inputs.
- A favourable contrast authorises only a separate Notebook 50 pre-registration.
- If M4 is not set-compatible with M3, the notebook says so explicitly rather than forcing an `M3 -> M4` subset interpretation.
- The completed Notebook 49 run remains immutable.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib scipy scikit-learn


## 1. Configuration and source resolution


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, os, platform, shutil, sys, zipfile
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
N49_ROOT=ROOT/'observed_print_classifier_v8_7_notebook49'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_7_notebook49r4'
SOURCE_N49_RUN_OVERRIDE=None
MIN_SUBREGION_PIXELS=2
MIN_GROUP_TARGET=10
MAX_REVIEW_CASES=40
DARK_THRESHOLD=200
SEED=49004

REQUIRED_N49=[
 '02_observable_lock/candidate_lock.json',
 '02_observable_lock/feature_schema.json',
 '03_truth_unlock/pixel_oracle_gap.parquet',
 '03_truth_unlock/region_oracle_gap.parquet',
 'notebook49_verdict.json','experiment_identity.json'
]
def complete_runs(root):
 hits=[]
 if Path(root).exists():
  for p in Path(root).iterdir():
   if p.is_dir() and all((p/r).exists() for r in REQUIRED_N49): hits.append(p)
 return sorted(hits,key=lambda p:p.stat().st_mtime)
if SOURCE_N49_RUN_OVERRIDE is not None:
 N49=Path(SOURCE_N49_RUN_OVERRIDE)
else:
 hits=complete_runs(N49_ROOT); N49=hits[-1] if hits else None
if N49 is None or not all((N49/r).exists() for r in REQUIRED_N49):
 raise RuntimeError('No complete Notebook 49 run found. Set SOURCE_N49_RUN_OVERRIDE to the completed run.')
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_7_notebook49r4'
OUT=OUTPUT_ROOT/RUN; INPUTS=OUT/'01_source_lock'; TABLES=OUT/'02_tables'; M4DIR=OUT/'03_m4_semantics'; VISUAL=OUT/'04_visual_audit'; AUDIT=OUT/'05_audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,TABLES,M4DIR,VISUAL,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
print('Notebook 49 source:',N49); print('Output:',OUT)


## 2. Verify and snapshot the completed Notebook 49 run


In [ ]:
def sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''): h.update(b)
 return h.hexdigest()
def ah(a):
 a=np.ascontiguousarray(a); return hashlib.sha256(str(a.dtype).encode()+str(a.shape).encode()+a.tobytes()).hexdigest()
def oh(v): return hashlib.sha256(json.dumps(v,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def atomic_json(path,v):
 tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(v,indent=2,sort_keys=True,default=str)); tmp.replace(path)
def require(ok,msg):
 if not ok:
  atomic_json(AUDIT/'fatal_failure.json',{'created_utc':datetime.now(timezone.utc).isoformat(),'message':msg})
  raise RuntimeError(msg)
lock49=json.loads((N49/'02_observable_lock/candidate_lock.json').read_text())
schema49=json.loads((N49/'02_observable_lock/feature_schema.json').read_text())
identity49=json.loads((N49/'experiment_identity.json').read_text())
verdict49=json.loads((N49/'notebook49_verdict.json').read_text())
pixels=pd.read_parquet(N49/'03_truth_unlock/pixel_oracle_gap.parquet')
regions=pd.read_parquet(N49/'03_truth_unlock/region_oracle_gap.parquet')
required={'page','source_group','y','x','region_id','component_id','diagnostic_class'}
require(required.issubset(pixels.columns),f'Notebook 49 pixel table missing {sorted(required-set(pixels.columns))}')
require(not pixels.duplicated(['page','y','x']).any(),'Notebook 49 pixel identities are duplicated.')
require(set(pixels.diagnostic_class.unique()).issubset({'PROTECT','REMOVE','RECONSTRUCT','UNCERTAIN'}),'Unexpected Notebook 49 class.')
payload=lock49.get('payload',{})
expected_pixel_sha=payload.get('observable_pixels_sha256')
# The truth table extends the locked observable table, so verify ordered identity rather than expecting equal file bytes.
identity_cols=['source_group','page','y','x','region_id','component_id']
current_identity=oh(pixels[identity_cols].astype(str).to_dict('records'))
require(current_identity==payload.get('ordered_identity_sha256'),'Notebook 49 ordered pixel identity no longer matches its candidate lock.')
source_snapshot=[]
for rel in REQUIRED_N49:
 p=N49/rel; source_snapshot.append({'relative_path':rel,'bytes':p.stat().st_size,'sha256':sha(p)})
pd.DataFrame(source_snapshot).to_csv(INPUTS/'source_notebook49_snapshot.csv',index=False)
for rel in ['02_observable_lock/candidate_lock.json','02_observable_lock/feature_schema.json','notebook49_verdict.json','experiment_identity.json']:
 dst=INPUTS/Path(rel).name; shutil.copy2(N49/rel,dst)
source_record={'source_notebook49':str(N49),'candidate_lock_id':lock49.get('lock_id'),'ordered_identity_sha256':current_identity,'pages':int(pixels.page.nunique()),'source_groups':int(pixels.source_group.nunique()),'pixels':int(len(pixels)),'regions':int(regions.region_id.nunique()),'source_verdict':verdict49.get('verdict'),'created_utc':datetime.now(timezone.utc).isoformat()}
atomic_json(OUT/'source_record.json',source_record)
print(json.dumps(source_record,indent=2))


## 3. Build truth-side connected subregions inside M3

Each class is connected independently using page-global 8-connectivity. Consequently, a subregion is class-pure by construction, while the parent M3 component may remain mixed. The analysis records the pure subregion's relationship to its parent component and to neighbouring truth classes.


In [ ]:
def bbox_stats(mask):
 ys,xs=np.where(mask); x0,x1=int(xs.min()),int(xs.max()+1); y0,y1=int(ys.min()),int(ys.max()+1); area=int(mask.sum()); w=x1-x0; h=y1-y0
 return x0,y0,w,h,area,float(area/max(w*h,1)),float(max(w,h)/max(min(w,h),1))
sub_rows=[]; membership=[]
for page,g in pixels.groupby('page',sort=False):
 H=int(g.y.max())+1; W=int(g.x.max())+1
 # Exact page dimensions are not needed for connectivity as long as all coordinates fit.
 class_map=np.full((H,W),'',dtype=object); class_map[g.y.to_numpy(int),g.x.to_numpy(int)]=g.diagnostic_class.to_numpy(object)
 parent_map=np.full((H,W),'',dtype=object); parent_map[g.y.to_numpy(int),g.x.to_numpy(int)]=g.region_id.astype(str).to_numpy(object)
 for cls in ['REMOVE','PROTECT','UNCERTAIN','RECONSTRUCT']:
  mask=class_map==cls
  n,lab,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8)
  for sid in range(1,n):
   m=lab==sid; x,y,w,h,area,fill,aspect=bbox_stats(m); ys,xs=np.where(m)
   parent_ids,parent_counts=np.unique(parent_map[m],return_counts=True); order=np.argsort(parent_counts)[::-1]; parent_ids=parent_ids[order]; parent_counts=parent_counts[order]
   dominant_parent=str(parent_ids[0]); parent_total=int((parent_map==dominant_parent).sum()); parent_same=int(parent_counts[0])
   ring=(cv2.dilate(m.astype(np.uint8),np.ones((3,3),np.uint8))>0)&~m
   neighbours=class_map[ring]; neighbour_counts={c:int(np.sum(neighbours==c)) for c in ['REMOVE','PROTECT','UNCERTAIN','RECONSTRUCT']}
   srid=f'{page}:{cls.lower()}:s{sid:05d}'
   sub_rows.append({'subregion_id':srid,'page':str(page),'source_group':str(g.source_group.iloc[0]),'diagnostic_class':cls,'x':x,'y':y,'w':w,'h':h,'area':area,'fill_ratio':fill,'aspect_ratio':aspect,'dominant_parent_region_id':dominant_parent,'parent_regions_touched':int(len(parent_ids)),'dominant_parent_share_of_subregion':float(parent_same/area),'subregion_share_of_dominant_parent':float(parent_same/max(parent_total,1)),'touches_other_truth_class':bool(sum(v for k,v in neighbour_counts.items() if k!=cls)>0),**{f'adjacent_{k.lower()}_pixels':v for k,v in neighbour_counts.items()}})
   membership.extend({'subregion_id':srid,'page':str(page),'y':int(yy),'x':int(xx),'diagnostic_class':cls} for yy,xx in zip(ys,xs))
subregions=pd.DataFrame(sub_rows); member=pd.DataFrame(membership)
require(len(subregions)>0,'No truth-side connected subregions were found.')
subregions.to_parquet(TABLES/'truth_connected_subregions.parquet',index=False)
member.to_parquet(TABLES/'truth_connected_subregion_membership.parquet',index=False)
print(subregions.groupby('diagnostic_class').agg(subregions=('subregion_id','size'),pixels=('area','sum'),median_area=('area','median'),max_area=('area','max')))


## 4. Coherence, fragmentation and parent-component mixture


In [ ]:
class_totals=pixels.groupby(['source_group','page','diagnostic_class'],as_index=False).size().rename(columns={'size':'class_pixels'})
coherence=[]
for cls,g in subregions.groupby('diagnostic_class'):
 areas=g.area.to_numpy(int); total=int(areas.sum()); kept=areas[areas>=MIN_SUBREGION_PIXELS]
 coherence.append({'diagnostic_class':cls,'pixels':total,'subregions':len(g),'singleton_subregions':int((areas==1).sum()),'singleton_pixels':int((areas==1).sum()),'pixels_in_run_ge_2':int(kept.sum()),'fraction_pixels_in_run_ge_2':float(kept.sum()/max(total,1)),'median_subregion_area':float(np.median(areas)),'p90_subregion_area':float(np.quantile(areas,.90)),'max_subregion_area':int(areas.max()),'pages':int(g.page.nunique()),'source_groups':int(g.source_group.nunique()),'fraction_touching_other_class':float(g.touches_other_truth_class.mean()),'fraction_crossing_multiple_parent_regions':float((g.parent_regions_touched>1).mean())})
coherence=pd.DataFrame(coherence); coherence.to_csv(TABLES/'subregion_coherence_summary.csv',index=False)
by_page=(subregions.groupby(['source_group','page','diagnostic_class'],as_index=False).agg(subregions=('subregion_id','size'),pixels=('area','sum'),singletons=('area',lambda x:int((x==1).sum())),median_area=('area','median'),max_area=('area','max'),touching_other_class=('touches_other_truth_class','mean')))
by_page.to_csv(TABLES/'subregion_coherence_by_page.csv',index=False)
by_group=(subregions.groupby(['source_group','diagnostic_class'],as_index=False).agg(pages=('page','nunique'),subregions=('subregion_id','size'),pixels=('area','sum'),singletons=('area',lambda x:int((x==1).sum())),median_area=('area','median'),max_area=('area','max'),touching_other_class=('touches_other_truth_class','mean')))
by_group.to_csv(TABLES/'subregion_coherence_by_source_group.csv',index=False)
parent_mix=(pixels.groupby(['source_group','page','region_id','diagnostic_class'],as_index=False).size().rename(columns={'size':'pixels'}).pivot_table(index=['source_group','page','region_id'],columns='diagnostic_class',values='pixels',fill_value=0).reset_index())
for c in ['PROTECT','REMOVE','RECONSTRUCT','UNCERTAIN']:
 if c not in parent_mix: parent_mix[c]=0
parent_mix['total']=parent_mix[['PROTECT','REMOVE','RECONSTRUCT','UNCERTAIN']].sum(axis=1)
parent_mix['operational_classes_present']=(parent_mix[['PROTECT','REMOVE','RECONSTRUCT']]>0).sum(axis=1)
parent_mix['truth_classes_present']=(parent_mix[['PROTECT','REMOVE','RECONSTRUCT','UNCERTAIN']]>0).sum(axis=1)
parent_mix['mixed_operational']=parent_mix.operational_classes_present>1
parent_mix['mixed_with_uncertain']=((parent_mix.UNCERTAIN>0)&((parent_mix[['PROTECT','REMOVE','RECONSTRUCT']]>0).any(axis=1)))
parent_mix.to_csv(TABLES/'parent_m3_component_mixture.csv',index=False)
print(coherence.to_string(index=False)); print(); print('Parent components:',len(parent_mix),'mixed operational:',int(parent_mix.mixed_operational.sum()),'mixed with uncertain:',int(parent_mix.mixed_with_uncertain.sum()))


## 5. Aggregate observable evidence to subregions

Only fields already present in the locked Notebook 49 pixel table are used. Identifiers, truth classes, raw coordinates and explicit truth-derived columns are excluded from candidate observable features.


In [ ]:
joined=member.merge(pixels,on=['page','y','x','diagnostic_class'],how='left',validate='one_to_one')
identifiers={'subregion_id','source_group','page','y','x','region_id','component_id','diagnostic_class'}
truth_terms=['truth','clean','class','m4','overlap','reconstruct','protect_fraction','remove_fraction','uncertain_fraction']
feature_candidates=[]
for c in joined.select_dtypes(include=[np.number,bool]).columns:
 if c in identifiers: continue
 if any(t in c.lower() for t in truth_terms): continue
 if pd.to_numeric(joined[c],errors='coerce').notna().sum()==0: continue
 feature_candidates.append(c)
agg_spec={}
for c in feature_candidates:
 agg_spec[c+'_mean']=(c,'mean'); agg_spec[c+'_std']=(c,'std'); agg_spec[c+'_max']=(c,'max'); agg_spec[c+'_min']=(c,'min')
features=joined.groupby(['subregion_id','source_group','page','diagnostic_class'],as_index=False).agg(**agg_spec)
features=features.merge(subregions,on=['subregion_id','source_group','page','diagnostic_class'],validate='one_to_one')
features.to_parquet(TABLES/'subregion_observable_features.parquet',index=False)
observable_columns=[c for c in features.select_dtypes(include=[np.number,bool]).columns if c not in {'x','y'} and not any(t in c.lower() for t in truth_terms)]
constant=[c for c in observable_columns if pd.to_numeric(features[c],errors='coerce').dropna().nunique()<=1]
observable_columns=[c for c in observable_columns if c not in constant]
pd.DataFrame({'observable_feature':observable_columns}).to_csv(TABLES/'subregion_observable_feature_allowlist.csv',index=False)
print('Subregions:',len(features),'observable aggregate features:',len(observable_columns),'constant excluded:',len(constant))


## 6. Observable-only univariate separability

This section compares:
- REMOVE versus all other non-RECONSTRUCT classes;
- PROTECT versus all other non-RECONSTRUCT classes;
- REMOVE versus UNCERTAIN;
- PROTECT versus UNCERTAIN.

The unit is a connected truth-side subregion with at least two pixels. Reported AUCs are descriptive one-feature separability, not fitted multivariate performance or promotion evidence.


In [ ]:
def auc_direction_free(v,y):
 v=np.asarray(v,float); y=np.asarray(y,bool); ok=np.isfinite(v)
 if ok.sum()<2 or y[ok].sum()==0 or (~y[ok]).sum()==0:return np.nan,np.nan
 a=float(roc_auc_score(y[ok],v[ok])); return a,max(a,1-a)
contrasts=[('REMOVE','OTHER',['PROTECT','REMOVE','UNCERTAIN']),('PROTECT','OTHER',['PROTECT','REMOVE','UNCERTAIN']),('REMOVE','UNCERTAIN',['REMOVE','UNCERTAIN']),('PROTECT','UNCERTAIN',['PROTECT','UNCERTAIN'])]
rows=[]; group_rows=[]
base=features[features.area>=MIN_SUBREGION_PIXELS].copy()
for target,reference,classes in contrasts:
 q=base[base.diagnostic_class.isin(classes)].copy(); y=(q.diagnostic_class==target).to_numpy()
 if target=='REMOVE' and reference=='OTHER': pass
 if target=='PROTECT' and reference=='OTHER': pass
 for feat in observable_columns:
  v=pd.to_numeric(q[feat],errors='coerce').to_numpy(float); ok=np.isfinite(v); a=v[ok&y]; b=v[ok&~y]
  if len(a)<2 or len(b)<2: continue
  u,p=mannwhitneyu(a,b,alternative='two-sided'); rbc=2*u/(len(a)*len(b))-1; auc,dfa=auc_direction_free(v,y)
  signs=[]; eligible=0
  for sg,z in q.groupby('source_group'):
   za=pd.to_numeric(z.loc[z.diagnostic_class==target,feat],errors='coerce').dropna(); zb=pd.to_numeric(z.loc[z.diagnostic_class!=target,feat],errors='coerce').dropna()
   if len(za)>=MIN_GROUP_TARGET and len(zb)>=MIN_GROUP_TARGET:
    eligible+=1; delta=float(za.median()-zb.median()); signs.append(np.sign(delta)); group_rows.append({'target':target,'reference':reference,'feature':feat,'source_group':sg,'target_n':len(za),'reference_n':len(zb),'median_delta':delta})
  agreement=max(sum(np.asarray(signs)>0),sum(np.asarray(signs)<0))/len(signs) if signs else np.nan
  rows.append({'target':target,'reference':reference,'feature':feat,'target_n':len(a),'reference_n':len(b),'target_median':float(np.median(a)),'reference_median':float(np.median(b)),'rank_biserial':float(rbc),'mannwhitney_p':float(p),'auc':auc,'direction_free_auc':dfa,'eligible_source_groups':eligible,'source_group_direction_agreement':agreement})
cols=['target','reference','feature','target_n','reference_n','target_median','reference_median','rank_biserial','mannwhitney_p','auc','direction_free_auc','eligible_source_groups','source_group_direction_agreement']
sep=pd.DataFrame(rows,columns=cols).sort_values(['target','reference','direction_free_auc'],ascending=[True,True,False]) if rows else pd.DataFrame(columns=cols)
sep.to_csv(TABLES/'subregion_univariate_separability.csv',index=False)
pd.DataFrame(group_rows,columns=['target','reference','feature','source_group','target_n','reference_n','median_delta']).to_csv(TABLES/'subregion_univariate_separability_by_source_group.csv',index=False)
stable=sep[(sep.direction_free_auc>=.65)&(sep.eligible_source_groups>=2)&(sep.source_group_direction_agreement>=.70)]
print('Stable descriptive contrasts:',len(stable)); print(stable.groupby(['target','reference']).head(12).to_string(index=False) if len(stable) else 'None at the descriptive gate.')


## 7. Resolve Notebook 45 and reconstruct M4 from its durable PNG outputs only


In [ ]:
# Resolve Notebook 45 from the source identity saved by Notebook 49.
n45_raw=identity49.get('source_notebook45') or payload.get('source_notebook45')
if not n45_raw:
 sr=INPUTS/'source_record.json'
 n45_raw=None
N45=Path(n45_raw) if n45_raw else None
require(N45 is not None and N45.exists(),'Notebook 49 identity does not resolve its source Notebook 45 run.')
for rel in ['mask_lock.json','source_record.json','tables/region_manifest.parquet']:
 require((N45/rel).exists(),f'Notebook 45 source missing {rel}')
manifest45=pd.read_parquet(N45/'tables/region_manifest.parquet')
source45=json.loads((N45/'source_record.json').read_text())
lock45=json.loads((N45/'mask_lock.json').read_text())
require({'page','region_id'}.issubset(manifest45.columns),'Notebook 45 region manifest lacks page/region_id.')
m4_masks={}; m45_inventory=[]
for page,g in manifest45.groupby('page',sort=False):
 page=str(page); pg=pixels[pixels.page.astype(str)==page]; require(len(pg)>0,f'{page}: Notebook 45 page absent from Notebook 49 pixels.')
 # Infer full shape from the exact V7.5 cache named by Notebook 45.
 status=pd.read_csv(Path(source45['v75'])/'tables/cache_status.csv'); cmap={str(r.page):Path(r.cache) for _,r in status.iterrows()}
 require(page in cmap,f'{page}: V7.5 cache unavailable for M4 shape.')
 with np.load(cmap[page],allow_pickle=False) as z:
  auth=z['mask']>0; shape=auth.shape
 m4=np.zeros(shape,bool)
 for _,r in g.iterrows():
  path=N45/'masks'/page/str(r.region_id).replace(':','_')/'M4.png'; require(path.exists(),f'Missing {path}')
  im=cv2.imread(str(path),0); require(im is not None and im.shape==shape,f'Unreadable or mismatched {path}'); m4|=im>0
 require(np.all(~m4|auth),f'{page}: M4 outside authoritative mask.')
 m4_masks[page]=m4; m45_inventory.append({'page':page,'m4_pixels':int(m4.sum()),'m4_sha256':ah(m4.astype(np.uint8)),'authoritative_pixels':int(auth.sum())})
pd.DataFrame(m45_inventory).to_csv(AUDIT/'m4_reconstruction_inventory.csv',index=False)
print('Notebook 45:',N45,'M4 pages:',len(m4_masks),'M4 pixels:',sum(x.sum() for x in m4_masks.values()))


## 8. Diagnose M4 set semantics

The audit compares M4 with:
- M3 membership reconstructed from Notebook 49's locked pixel roster;
- each Notebook 49 truth-side class;
- observed dark material;
- clean printed ink and its one-pixel tolerance;
- the authoritative mask.

No assumption is made that M4 must be a subset of M3.


In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def imap(folder): return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def printed_ink(im): return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
clean_map=imap(ROOT/'source'/'clean')
# Observed images follow Notebook 45's V8.2 source record.
source_v82=Path(source45['source_v82']); obs_base=source_v82/'whole_pages'
sem=[]
for page,m4 in m4_masks.items():
 pg=pixels[pixels.page.astype(str)==page]; m3=np.zeros(m4.shape,bool); m3[pg.y.to_numpy(int),pg.x.to_numpy(int)]=True
 require(page in clean_map,f'{page}: clean page unavailable for M4 semantics.')
 clean=cv2.imread(str(clean_map[page]),0); observed=cv2.imread(str(obs_base/page/'observed.png'),0)
 require(clean is not None and observed is not None and clean.shape==m4.shape and observed.shape==m4.shape,f'{page}: clean/observed shape mismatch.')
 ci=printed_ink(clean); near=cv2.dilate(ci.astype(np.uint8),np.ones((3,3),np.uint8))>0; od=observed<DARK_THRESHOLD
 classes={c:np.zeros(m4.shape,bool) for c in ['PROTECT','REMOVE','RECONSTRUCT','UNCERTAIN']}
 for c in classes:
  q=pg[pg.diagnostic_class==c]; classes[c][q.y.to_numpy(int),q.x.to_numpy(int)]=True
 inter=lambda a,b:int((a&b).sum())
 row={'page':page,'source_group':str(pg.source_group.iloc[0]),'M3_pixels':int(m3.sum()),'M4_pixels':int(m4.sum()),'M3_M4_intersection':inter(m3,m4),'M4_inside_M3_fraction':float(inter(m3,m4)/max(m4.sum(),1)),'M3_covered_by_M4_fraction':float(inter(m3,m4)/max(m3.sum(),1)),'M4_observed_dark_pixels':inter(m4,od),'M4_observed_dark_fraction':float(inter(m4,od)/max(m4.sum(),1)),'M4_clean_ink_pixels':inter(m4,ci),'M4_clean_ink_fraction':float(inter(m4,ci)/max(m4.sum(),1)),'M4_clean_ink_near_pixels':inter(m4,near),'M4_clean_ink_near_fraction':float(inter(m4,near)/max(m4.sum(),1))}
 for c,m in classes.items(): row[f'M4_{c.lower()}_intersection']=inter(m4,m)
 sem.append(row)
sem=pd.DataFrame(sem); sem.to_csv(M4DIR/'m4_semantics_by_page.csv',index=False)
pooled={}
for c in ['M3_pixels','M4_pixels','M3_M4_intersection','M4_observed_dark_pixels','M4_clean_ink_pixels','M4_clean_ink_near_pixels','M4_protect_intersection','M4_remove_intersection','M4_reconstruct_intersection','M4_uncertain_intersection']:
 pooled[c]=int(sem[c].sum())
pooled.update({'M4_inside_M3_fraction':pooled['M3_M4_intersection']/max(pooled['M4_pixels'],1),'M3_covered_by_M4_fraction':pooled['M3_M4_intersection']/max(pooled['M3_pixels'],1),'M4_observed_dark_fraction':pooled['M4_observed_dark_pixels']/max(pooled['M4_pixels'],1),'M4_clean_ink_fraction':pooled['M4_clean_ink_pixels']/max(pooled['M4_pixels'],1),'M4_clean_ink_near_fraction':pooled['M4_clean_ink_near_pixels']/max(pooled['M4_pixels'],1)})
if pooled['M3_M4_intersection']==0:
 semantic_verdict='M4_DISJOINT_FROM_M3_ON_COMPLETED_ROSTER'
elif pooled['M4_inside_M3_fraction']>=.99:
 semantic_verdict='M4_APPROXIMATE_SUBSET_OF_M3'
elif pooled['M3_covered_by_M4_fraction']>=.99:
 semantic_verdict='M3_APPROXIMATE_SUBSET_OF_M4'
else:
 semantic_verdict='M4_AND_M3_PARTIALLY_OVERLAP'
pooled['semantic_verdict']=semantic_verdict
atomic_json(M4DIR/'m4_semantics_summary.json',pooled)
print(json.dumps(pooled,indent=2))


## 9. Visual audit queues


In [ ]:
# Queue large coherent target regions, mixed-parent examples, and M4/M3 diagnostic pages.
queues=[]
for cls in ['REMOVE','PROTECT','UNCERTAIN']:
 q=subregions[(subregions.diagnostic_class==cls)&(subregions.area>=MIN_SUBREGION_PIXELS)].sort_values(['area','subregion_share_of_dominant_parent'],ascending=False).head(MAX_REVIEW_CASES).copy(); q['queue']=cls.lower()+'_coherent'; queues.append(q)
mix_ids=set(parent_mix.loc[parent_mix.mixed_with_uncertain|parent_mix.mixed_operational,'region_id'].astype(str))
q=subregions[subregions.dominant_parent_region_id.astype(str).isin(mix_ids)].sort_values('area',ascending=False).head(MAX_REVIEW_CASES).copy(); q['queue']='mixed_parent'; queues.append(q)
review=pd.concat(queues,ignore_index=True) if queues else pd.DataFrame(); review.to_csv(TABLES/'subregion_visual_audit_queue.csv',index=False)
# M4 page queue prioritises largest XOR then largest M4.
m4q=sem.assign(xor=lambda x:x.M3_pixels+x.M4_pixels-2*x.M3_M4_intersection).sort_values(['xor','M4_pixels'],ascending=False).head(MAX_REVIEW_CASES)
m4q.to_csv(TABLES/'m4_visual_audit_queue.csv',index=False)
print('Subregion review rows:',len(review),'M4 review pages:',len(m4q))


## 10. Evidence verdict


In [ ]:
coh=coherence.set_index('diagnostic_class') if len(coherence) else pd.DataFrame()
def coherent(cls): return bool(cls in coh.index and coh.loc[cls,'pixels']>0 and coh.loc[cls,'fraction_pixels_in_run_ge_2']>=.50 and coh.loc[cls,'source_groups']>=2)
def stable_count(cls): return int(((stable.target==cls)&(stable.reference.isin(['OTHER','UNCERTAIN']))).sum()) if len(stable) else 0
checks={'source_notebook49_identity_verified':True,'completed_notebook49_reused_without_upstream_rebuild':True,'remove_present':bool((pixels.diagnostic_class=='REMOVE').any()),'protect_present':bool((pixels.diagnostic_class=='PROTECT').any()),'reconstruct_present':bool((pixels.diagnostic_class=='RECONSTRUCT').any()),'remove_connected_coherence':coherent('REMOVE'),'protect_connected_coherence':coherent('PROTECT'),'remove_stable_univariate_contrasts':stable_count('REMOVE'),'protect_stable_univariate_contrasts':stable_count('PROTECT'),'m4_semantic_verdict':semantic_verdict}
if not checks['source_notebook49_identity_verified']:
 verdict='INVALID_RUN'
elif not checks['remove_present'] and not checks['protect_present']:
 verdict='NO_OPERATIONAL_EXCLUSION_TARGET'
elif checks['remove_connected_coherence'] and checks['remove_stable_univariate_contrasts']>0 and checks['protect_connected_coherence'] and checks['protect_stable_univariate_contrasts']>0:
 verdict='REMOVE_AND_PROTECT_SUPPORT_SEPARATE_NOTEBOOK50_PREREGISTRATION'
elif (checks['remove_connected_coherence'] and checks['remove_stable_univariate_contrasts']>0) or (checks['protect_connected_coherence'] and checks['protect_stable_univariate_contrasts']>0):
 verdict='ONE_TARGET_SUPPORTS_NARROW_NOTEBOOK50_PREREGISTRATION'
else:
 verdict='COHERENCE_OR_OBSERVABLE_SEPARABILITY_INSUFFICIENT'
record={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'49R4','verdict':verdict,'checks':checks,'interpretation':'This verdict concerns connected-subregion coherence and descriptive observable separability only. It does not train or authorise M5.','m4_interpretation':semantic_verdict,'next_action':{'REMOVE_AND_PROTECT_SUPPORT_SEPARATE_NOTEBOOK50_PREREGISTRATION':'Pre-register asymmetric REMOVE and PROTECT targets, grouped validation and fail-closed UNCERTAIN handling.','ONE_TARGET_SUPPORTS_NARROW_NOTEBOOK50_PREREGISTRATION':'Pre-register only the supported target; retain all other M3 treatment unchanged.','COHERENCE_OR_OBSERVABLE_SEPARABILITY_INSUFFICIENT':'Do not train M5 from these targets; retain M3 and reconsider evidence/labels.','NO_OPERATIONAL_EXCLUSION_TARGET':'Stop the exclusion-target programme.','INVALID_RUN':'Correct identity/integrity failure first.'}[verdict]}
atomic_json(OUT/'notebook49r4_verdict.json',record); print(json.dumps(record,indent=2))


## 11. Summary and curated upload package


In [ ]:
environment={'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}; atomic_json(OUT/'software_environment.json',environment)
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'49R4-connected-subregion-and-m4-semantics','development_only':True,'model_fitting':False,'promotion_authority':False,'source_notebook49':str(N49),'source_notebook49_lock_id':lock49.get('lock_id'),'source_notebook45':str(N45),'pages':int(pixels.page.nunique()),'source_groups':int(pixels.source_group.nunique()),'pixels':int(len(pixels)),'subregions':int(len(subregions)),'observable_features':observable_columns,'software':environment}; atomic_json(OUT/'experiment_identity.json',identity)
class_counts=pixels.diagnostic_class.value_counts().to_dict()
lines=['NOTEBOOK 49R4: CONNECTED-SUBREGION AND M4 SEMANTICS AUDIT','',f'Source Notebook 49: {N49}',f'Source lock: {lock49.get("lock_id")}',f'Pages: {pixels.page.nunique()}',f'Source groups: {pixels.source_group.nunique()}',f'M3 pixels: {len(pixels)}',f'Truth-side connected subregions: {len(subregions)}','',*[f'{k}: {v}' for k,v in class_counts.items()],'',f'M4 semantic verdict: {semantic_verdict}',f'49R4 verdict: {verdict}',record['next_action'],'','No Notebook 45 inventory or upstream mask construction was rerun.','No model was fitted or threshold selected.']
(OUT/'executive_summary.txt').write_text(chr(10).join(lines))
required=[OUT/'executive_summary.txt',OUT/'notebook49r4_verdict.json',OUT/'experiment_identity.json',OUT/'source_record.json',TABLES/'subregion_coherence_summary.csv',TABLES/'subregion_coherence_by_page.csv',TABLES/'subregion_coherence_by_source_group.csv',TABLES/'parent_m3_component_mixture.csv',TABLES/'subregion_univariate_separability.csv',TABLES/'subregion_univariate_separability_by_source_group.csv',TABLES/'subregion_visual_audit_queue.csv',TABLES/'m4_visual_audit_queue.csv',M4DIR/'m4_semantics_by_page.csv',M4DIR/'m4_semantics_summary.json',AUDIT/'m4_reconstruction_inventory.csv',INPUTS/'source_notebook49_snapshot.csv']
for p in required: require(p.exists(),f'Missing required review artefact {p}')
for p in required:
 dst=UPLOAD/p.relative_to(OUT); dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(p,dst)
(UPLOAD/'README.txt').write_text('Notebook 49R4 curated review package. Heavy parquet tables remain in the run directory and are identified by experiment_identity.json.'+chr(10))
inv=[]
for p in sorted(UPLOAD.rglob('*')):
 if p.is_file(): inv.append({'relative_path':str(p.relative_to(UPLOAD)),'bytes':p.stat().st_size,'sha256':sha(p)})
pd.DataFrame(inv).to_csv(UPLOAD/'upload_manifest.csv',index=False); (UPLOAD/'upload_file_list.txt').write_text(chr(10).join(x['relative_path'] for x in inv))
zip_path=UPLOAD/'notebook49r4_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
 for p in sorted(UPLOAD.rglob('*')):
  if p.is_file() and p!=zip_path: z.write(p,p.relative_to(UPLOAD))
print(chr(10).join(lines)); print('Upload bundle:',zip_path)


## Outputs to inspect first

1. `executive_summary.txt`
2. `notebook49r4_verdict.json`
3. `02_tables/subregion_coherence_summary.csv`
4. `02_tables/subregion_coherence_by_source_group.csv`
5. `02_tables/parent_m3_component_mixture.csv`
6. `02_tables/subregion_univariate_separability.csv`
7. `03_m4_semantics/m4_semantics_summary.json`
8. `03_m4_semantics/m4_semantics_by_page.csv`
9. `02_tables/subregion_visual_audit_queue.csv`
10. `upload/notebook49r4_upload_bundle.zip`

A positive result permits only a separate, pre-registered Notebook 50 experiment. It does not select a model, operating threshold or M5 mask.
